In [ ]:
import os, json, pickle, glob
import numpy as np
import matplotlib.pyplot as plt

SEEDS = [42]
SCALE_SET = "all"
DATASET = "cifar10"

ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    ROOT = os.path.dirname(ROOT)
CK = os.path.join(ROOT, "final-checkpoints", DATASET)
FIGS = os.path.join(ROOT, "final-results", "figures")
os.makedirs(FIGS, exist_ok=True)

SCALE_GROUPS = {
    "base": ["small", "deeper-small", "medium", "large"],
    "extended": ["vlarge", "huge"],
    "all": ["small", "deeper-small", "medium", "large", "vlarge", "huge", "vhuge"],
}
SCALES = SCALE_GROUPS[SCALE_SET]
CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
ARCH = {"ternary": "TLGN", "binary": "BLGN"}

def save(fig, name):
    for ext in ["svg", "pdf"]:
        fig.savefig(os.path.join(FIGS, name + "." + ext), bbox_inches="tight")

print(DATASET, "| seeds", SEEDS, "| scales", SCALES)

In [ ]:
rows = [json.loads(l) for l in open(os.path.join(ROOT, "final-checkpoints", "results.jsonl")) if l.strip()]
rows = [r for r in rows if r["dataset"] == DATASET and r["seed"] in SEEDS and r.get("status") == "ok"]

def scalars(scale, arch, key):
    return np.array([r[key] for r in rows if r["scale"] == scale and r["arch"] == ARCH[arch]], dtype=float)

print("scale          arch   n  soft_test      circuit_test   gap(pp)       unknown%")
for scale in SCALES:
    for arch in ["binary", "ternary"]:
        st, ct = scalars(scale, arch, "soft_test_acc") * 100, scalars(scale, arch, "circuit_test_acc") * 100
        uk = scalars(scale, arch, "unknown_pct") * 100
        if not len(st):
            continue
        def f(v):
            return str(round(v.mean(), 2)) + ("±" + str(round(v.std(ddof=1), 2)) if len(v) > 1 else "")
        print(scale.ljust(14), ARCH[arch].ljust(6), str(len(st)).ljust(2),
              f(st).ljust(14), f(ct).ljust(14), f(st - ct).ljust(13), f(uk))

In [ ]:
import sys
sys.path.insert(0, ROOT)
from pst_dtlgn.data.pipeline import BinaryPipeline, TernaryPipeline, load_cifar10, load_cifar100
from pst_dtlgn.network.harden import TernaryLearnedCircuit
from pst_dtlgn.binary_baseline import BinaryLearnedCircuit, BIN_TRUTH_TABLES

loader = load_cifar10 if DATASET == "cifar10" else load_cifar100
data = loader(os.path.join(ROOT, "data", DATASET))
bin_test = BinaryPipeline(resolution=4).fit(data["train_x"]).transform(data["test_x"], mode="hard")
ter_test = TernaryPipeline(resolution=4).fit(data["train_x"]).transform(data["test_x"], mode="hard")
y_test = data["test_y"]
N_CLASSES = 10 if DATASET == "cifar10" else 100
print("test set", data["test_x"].shape)

In [ ]:
def ternary_forward(c, X):
    h = X.astype(np.int8)
    for tt, conn in zip(c.truth_tables, c.connections):
        a = h[:, conn[:, 0]].astype(np.int32); b = h[:, conn[:, 1]].astype(np.int32)
        h = tt.reshape(-1)[np.arange(tt.shape[0], dtype=np.int64)[None, :] * 9 + (a + 1) * 3 + (b + 1)].astype(np.int8)
    return h

def binary_forward(c, X):
    h = X.astype(np.int8); bt = np.asarray(BIN_TRUTH_TABLES).reshape(-1)
    for gidx, conn in zip(c.gate_indices, c.connections):
        gidx = np.asarray(gidx, dtype=np.int64)
        a = h[:, conn[:, 0]].astype(np.int32); b = h[:, conn[:, 1]].astype(np.int32)
        h = bt[gidx[None, :] * 4 + (a * 2 + b)].astype(np.int8)
    return h

_cache = {}

def per_class(scale, arch, seed, chunk=2000):
    key = (scale, arch, seed)
    if key in _cache:
        return _cache[key]
    p = os.path.join(CK, scale + "_" + arch + "_seed" + str(seed) + "_harden.pkl")
    if not os.path.exists(p):
        _cache[key] = None
        return None
    hr = pickle.load(open(p, "rb"))
    circ = TernaryLearnedCircuit(hr) if arch == "ternary" else BinaryLearnedCircuit(hr)
    xs = np.asarray(ter_test if arch == "ternary" else bin_test, dtype=np.int8)
    fwd = ternary_forward if arch == "ternary" else binary_forward
    preds = np.empty(len(xs), dtype=np.int32)
    for s0 in range(0, len(xs), chunk):
        s1 = min(s0 + chunk, len(xs))
        out = fwd(circ, xs[s0:s1])
        g = out.shape[1] // N_CLASSES
        preds[s0:s1] = out.reshape(out.shape[0], N_CLASSES, g).sum(-1).argmax(-1)
    acc = np.array([(preds[y_test == c] == c).mean() for c in range(N_CLASSES)])
    _cache[key] = acc
    del hr, circ
    return acc

def per_class_seeds(scale, arch):
    got = [a for a in (per_class(scale, arch, s) for s in SEEDS) if a is not None]
    return np.array(got) if got else None

In [ ]:
have = [s for s in SCALES if per_class_seeds(s, "ternary") is not None]
missing = [s for s in SCALES if s not in have]
if missing:
    print("no checkpoints for:", missing, "- run 00_cifar_scaling_battery.ipynb to add them")
print("plotting:", have)

In [ ]:
if have:
    x = np.arange(len(CLASSES))
    w = 0.8 / (2 * len(have))
    fig, ax = plt.subplots(figsize=(1.6 * len(CLASSES), 6))
    blues = plt.cm.Blues(np.linspace(0.45, 0.9, len(have)))
    oranges = plt.cm.Oranges(np.linspace(0.45, 0.9, len(have)))
    for i, scale in enumerate(have):
        for j, (arch, cols) in enumerate([("binary", blues), ("ternary", oranges)]):
            P = per_class_seeds(scale, arch)
            if P is None:
                continue
            m = P.mean(0) * 100
            err = (P.max(0) - P.min(0)) * 100 / 2 if len(P) > 1 else None
            off = (2 * i + j - (2 * len(have) - 1) / 2) * w
            ax.bar(x + off, m, w, yerr=err, capsize=2, color=cols[i],
                   edgecolor="white", linewidth=0.4, label=ARCH[arch] + "-" + scale)
    ax.set_xticks(x); ax.set_xticklabels(CLASSES, rotation=30, ha="right")
    ax.set_ylabel("Circuit accuracy (%)")
    ax.set_title("Per-class circuit accuracy — DLGN vs TLGN across scales")
    ax.legend(fontsize=8, ncol=2, frameon=False)
    fig.tight_layout()
    save(fig, "cifar_per_class_accuracy")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.8))
for arch, col in [("binary", "tab:blue"), ("ternary", "tab:orange")]:
    xs, soft_m, circ_m, soft_e, circ_e = [], [], [], [], []
    for scale in SCALES:
        n = scalars(scale, arch, "neurons")
        st, ct = scalars(scale, arch, "soft_test_acc") * 100, scalars(scale, arch, "circuit_test_acc") * 100
        if not len(st):
            continue
        xs.append(n[0]); soft_m.append(st.mean()); circ_m.append(ct.mean())
        soft_e.append(st.std(ddof=1) if len(st) > 1 else 0.0)
        circ_e.append(ct.std(ddof=1) if len(ct) > 1 else 0.0)
    if not xs:
        continue
    ax.errorbar(xs, soft_m, yerr=soft_e, marker="o", color=col, lw=1.9, capsize=3,
                label=ARCH[arch] + " soft")
    ax.errorbar(xs, circ_m, yerr=circ_e, marker="s", ls="--", color=col, lw=1.5, capsize=3,
                alpha=0.75, label=ARCH[arch] + " circuit")
ax.set_xscale("log"); ax.set_xlabel("neurons"); ax.set_ylabel("test accuracy (%)")
ax.set_title("Scaling: soft network vs hardened circuit")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25, lw=0.5)
fig.tight_layout()
save(fig, "cifar_scaling_curve")